
# Proyecto ML para Transformadores Eléctricos

## Notebook 02 — Integración con Amazon S3 y selección de modelos

### Objetivo

Construir un flujo reproducible de Machine Learning que conecte SageMaker Studio con Amazon S3.

Este notebook representa el primer punto del laboratorio en el que AWS participa directamente en el flujo de datos y artefactos.

### Flujo completo

```text
Notebook 01
Generación y exploración
        │
        ▼
transformadores.csv
        │
        ▼
SageMaker Studio
        │
        ▼
Amazon S3 / data/raw
        │
        ▼
Carga desde S3
        │
        ▼
Preparación
        │
        ├───────────────┐
        ▼               ▼
data/train         data/test
        │               │
        └───────┬───────┘
                ▼
        Entrenamiento local
                │
                ▼
        Evaluación y comparación
                │
                ▼
        Selección de modelos
                │
        ┌───────┴────────┐
        ▼                ▼
     model/           output/
    modelos          métricas
                    predicciones
```

### Importante

El entrenamiento de este notebook se ejecuta dentro de JupyterLab.

Todavía **no** se utilizan SageMaker Training Jobs ni endpoints. Esa separación es intencional y permite enseñar primero el flujo fundamental de datos, entrenamiento, evaluación y persistencia antes de introducir servicios administrados.

> El Notebook 01 permanece exclusivamente enfocado en generación y exploración de datos. Este Notebook 02 es el primero que incorpora AWS.



## 1. Estructura esperada del proyecto en S3

Al finalizar el notebook tendremos:

```text
siemens-ml-lab-aiarch-2026/
│
├── data/
│   ├── raw/
│   │   └── transformadores.csv
│   ├── train/
│   │   └── train.csv
│   └── test/
│       └── test.csv
│
├── model/
│   ├── modelo_01_*.joblib
│   ├── modelo_02_*.joblib
│   └── mejor_modelo.joblib
│
└── output/
    ├── metricas_modelos.csv
    └── predicciones_test.csv
```

En Amazon S3 no existen carpetas físicas como en un sistema de archivos tradicional. Lo que visualmente aparece como una carpeta corresponde a un **prefijo dentro de la clave de un objeto**.

Por ejemplo:

```text
data/raw/transformadores.csv
```

es la clave de un objeto dentro del bucket.

### Convención utilizada

| Prefijo | Propósito |
|---|---|
| `data/raw/` | Fuente original |
| `data/train/` | Datos de entrenamiento |
| `data/test/` | Datos de prueba |
| `model/` | Artefactos de modelos |
| `output/` | Métricas y predicciones |


In [ ]:
# ==========================================
# 2. Librerías
# ==========================================

import boto3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Librerías cargadas correctamente.")


## 3. Configuración del proyecto

No se escriben credenciales AWS en el notebook.

SageMaker Studio obtiene las credenciales temporales a través del **IAM Role asociado al entorno de ejecución**.

Esto es importante desde el punto de vista de seguridad:

```text
JupyterLab
    │
    ▼
IAM Role de SageMaker
    │
    ▼
Permisos S3
    │
    ▼
Amazon S3
```

### Bucket utilizado

```text
siemens-ml-lab-aiarch-2026
```

### Objetos principales

```text
data/raw/transformadores.csv
data/train/train.csv
data/test/test.csv
model/
output/
```


In [ ]:
# ==========================================
# Configuración AWS / S3
# ==========================================

BUCKET = "siemens-ml-lab-aiarch-2026"
REGION = boto3.Session().region_name or "us-east-2"

S3_RAW = "data/raw/transformadores.csv"
S3_TRAIN = "data/train/train.csv"
S3_TEST = "data/test/test.csv"
S3_MODEL_PREFIX = "model/"
S3_OUTPUT_PREFIX = "output/"

LOCAL_DATA = "transformadores.csv"
LOCAL_TRAIN = "train.csv"
LOCAL_TEST = "test.csv"

LOCAL_MODEL_DIR = Path("model_artifacts")
LOCAL_OUTPUT_DIR = Path("outputs")

LOCAL_MODEL_DIR.mkdir(exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(exist_ok=True)

session = boto3.Session(region_name=REGION)
s3 = session.client("s3")

print(f"Bucket: {BUCKET}")
print(f"Región: {REGION}")


## 4. Verificar acceso al bucket

Primero se comprueba que el rol de SageMaker Studio puede consultar el bucket.

Esta operación no modifica información. Solamente verifica que el entorno puede acceder al bucket.

Si aparece `AccessDenied`, debe revisarse el IAM Role de SageMaker y sus permisos sobre el bucket.


In [ ]:
s3.head_bucket(Bucket=BUCKET)
print(f"Acceso correcto al bucket: {BUCKET}")


## 5. Cargar el dataset original a S3

El archivo `transformadores.csv` fue generado en el Notebook 01.

En esta etapa se realiza la primera transferencia:

```text
Archivo local
     │
     │ boto3 / upload_file
     ▼
S3 / data/raw
```

La ubicación final será:

```text
s3://siemens-ml-lab-aiarch-2026/data/raw/transformadores.csv
```

### Nota sobre el error anterior

La versión anterior del notebook fallaba en esta operación porque el IAM Role no tenía permiso `s3:PutObject`.

Ese permiso ya fue incorporado y probado correctamente. Por eso el notebook definitivo **no contiene las celdas temporales de prueba de permisos** (`put_object` y `delete_object`).


In [ ]:
if not Path(LOCAL_DATA).exists():
    raise FileNotFoundError(
        f"No se encontró {LOCAL_DATA}. "
        "Coloca el CSV generado en el notebook anterior en el directorio de trabajo."
    )

s3.upload_file(LOCAL_DATA, BUCKET, S3_RAW)

print("Dataset cargado correctamente en S3.")
print(f"s3://{BUCKET}/{S3_RAW}")


## 6. Cargar la información directamente desde S3

A partir de este punto, S3 se convierte en la fuente oficial del dataset para el modelado.

El archivo local solamente se utiliza en la etapa anterior para realizar la carga inicial.

```text
Amazon S3
   │
   │ get_object
   ▼
Bytes del objeto
   │
   │ pandas.read_csv
   ▼
DataFrame
```

Esto permite demostrar un patrón muy utilizado en arquitecturas de Machine Learning: **almacenamiento desacoplado del entorno de cómputo**.


In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=S3_RAW)
df = pd.read_csv(obj["Body"])

print(f"Dataset cargado desde S3: {df.shape[0]} filas y {df.shape[1]} columnas")
display(df.head())


## 7. Validación básica del dataset

Antes de entrenar cualquier modelo se verifica que la información cargada desde S3 sea consistente.

Se revisan:

- nombres de columnas;
- tipos de datos;
- valores faltantes;
- estadísticas descriptivas.

La validación evita que un problema de calidad de datos sea confundido posteriormente con un problema del algoritmo de Machine Learning.


In [ ]:
print("Tipos de datos:")
display(df.dtypes)

print("Valores faltantes por variable:")
display(df.isna().sum())


## 8. Separar variables predictoras y variable objetivo

El problema es de **regresión supervisada**.

La variable que queremos predecir es:

```text
temperatura_transformador
```

Por tanto:

```text
X = variables predictoras
y = variable objetivo
```

Las variables predictoras representan condiciones ambientales y operativas del transformador.


In [ ]:
TARGET = "temperatura_transformador"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Variables predictoras:")
print(list(X.columns))
print(f"\nVariable objetivo: {TARGET}")


## 9. División entrenamiento / prueba

Se utilizará una división:

- **80 % entrenamiento**
- **20 % prueba**

El conjunto de entrenamiento se utiliza para ajustar los modelos.

El conjunto de prueba se reserva para evaluar cómo se comportan sobre datos que no participaron en el entrenamiento.

Se utiliza `random_state=42` para que la división sea reproducible.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

train = X_train.copy()
train[TARGET] = y_train

test = X_test.copy()
test[TARGET] = y_test

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")


## 10. Guardar Train y Test en S3

Una vez realizada la división, los dos conjuntos se persisten en S3.

```text
data/
├── raw/
│   └── transformadores.csv
├── train/
│   └── train.csv
└── test/
    └── test.csv
```

Esto permite que etapas posteriores del laboratorio puedan utilizar directamente los datos preparados sin tener que repetir la división.


In [ ]:
train.to_csv(LOCAL_TRAIN, index=False)
test.to_csv(LOCAL_TEST, index=False)

s3.upload_file(LOCAL_TRAIN, BUCKET, S3_TRAIN)
s3.upload_file(LOCAL_TEST, BUCKET, S3_TEST)

print("Train y Test cargados en S3.")
print(f"s3://{BUCKET}/{S3_TRAIN}")
print(f"s3://{BUCKET}/{S3_TEST}")


## 11. Comprobar los objetos creados en S3

Se listan los objetos bajo `data/`.

La intención es obtener evidencia de que S3 ya contiene:

1. la fuente original;
2. el conjunto de entrenamiento;
3. el conjunto de prueba.


In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="data/")

for item in response.get("Contents", []):
    print(f"{item['Key']} - {item['Size']} bytes")


## 12. Entrenamiento y comparación de modelos

Se compararán cuatro algoritmos de regresión:

1. Regresión lineal.
2. Árbol de decisión.
3. Random Forest.
4. Gradient Boosting.

Todos utilizarán el mismo conjunto de entrenamiento y serán evaluados sobre el mismo conjunto de prueba.

Esto hace que la comparación sea consistente.

### Criterio de selección

Se calcularán:

- **MAE:** error absoluto medio;
- **RMSE:** raíz del error cuadrático medio;
- **R²:** coeficiente de determinación.

El criterio principal de selección será **RMSE**, buscando el menor valor.


In [ ]:
models = {
    "regresion_lineal": LinearRegression(),
    "arbol_decision": DecisionTreeRegressor(
        max_depth=6,
        random_state=42
    ),
    "random_forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    trained_models[name] = model

    results.append({
        "modelo": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

metrics_df = (
    pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(metrics_df)


## 13. Selección de los mejores modelos

Los modelos se ordenan por RMSE.

```text
Menor RMSE
    ↓
Menor error de predicción
```

Se conservarán los dos mejores modelos para disponer de una alternativa además del ganador.

El primer modelo de la clasificación será considerado el **mejor modelo del experimento**.


In [ ]:
TOP_N = 2

best_models = metrics_df.head(TOP_N)["modelo"].tolist()

print("Mejores modelos:")
for position, name in enumerate(best_models, start=1):
    print(f"{position}. {name}")


## 14. Guardar los modelos localmente

Los modelos entrenados se convierten en artefactos mediante `joblib`.

Esto permite separar:

```text
Modelo en memoria
      ↓
Serialización
      ↓
Archivo .joblib
      ↓
Carga posterior
```

Los archivos se guardan primero localmente y posteriormente se transfieren a S3.


In [ ]:
model_files = []

for rank, name in enumerate(best_models, start=1):
    filename = LOCAL_MODEL_DIR / f"modelo_{rank}_{name}.joblib"
    joblib.dump(trained_models[name], filename)
    model_files.append((name, filename))
    print(f"Guardado: {filename}")

best_model_name = best_models[0]
best_model_path = LOCAL_MODEL_DIR / "mejor_modelo.joblib"

joblib.dump(trained_models[best_model_name], best_model_path)

print(f"\nMejor modelo: {best_model_name}")
print(f"Artefacto: {best_model_path}")


## 15. Cargar los mejores modelos a S3

Los artefactos seleccionados se almacenarán en:

```text
s3://siemens-ml-lab-aiarch-2026/model/
```

La carpeta `model/` queda separada de los datos y de los resultados.

Esto facilita una arquitectura posterior en la que un servicio de inferencia pueda recuperar el modelo desde S3.


In [ ]:
for name, filename in model_files:
    s3_key = f"{S3_MODEL_PREFIX}{filename.name}"
    s3.upload_file(str(filename), BUCKET, s3_key)
    print(f"{name} → s3://{BUCKET}/{s3_key}")

s3.upload_file(
    str(best_model_path),
    BUCKET,
    f"{S3_MODEL_PREFIX}mejor_modelo.joblib"
)

print("Artefactos de modelo cargados correctamente en S3.")


## 16. Guardar métricas y predicciones

Además de almacenar los modelos, se conservan evidencias del experimento.

### Métricas

```text
output/metricas_modelos.csv
```

Incluye MAE, RMSE y R² para cada algoritmo.

### Predicciones

```text
output/predicciones_test.csv
```

Incluye:

- variables del conjunto de prueba;
- temperatura real;
- temperatura predicha;
- error absoluto.

Esto permite conservar trazabilidad del experimento.


In [ ]:
metrics_path = LOCAL_OUTPUT_DIR / "metricas_modelos.csv"
metrics_df.to_csv(metrics_path, index=False)

best_model = trained_models[best_model_name]
test_predictions = best_model.predict(X_test)

predictions_df = X_test.copy()
predictions_df["temperatura_real"] = y_test.values
predictions_df["temperatura_predicha"] = test_predictions
predictions_df["error_absoluto"] = np.abs(
    predictions_df["temperatura_real"]
    - predictions_df["temperatura_predicha"]
)

predictions_path = LOCAL_OUTPUT_DIR / "predicciones_test.csv"
predictions_df.to_csv(predictions_path, index=False)

s3.upload_file(
    str(metrics_path),
    BUCKET,
    f"{S3_OUTPUT_PREFIX}metricas_modelos.csv"
)

s3.upload_file(
    str(predictions_path),
    BUCKET,
    f"{S3_OUTPUT_PREFIX}predicciones_test.csv"
)

print("Métricas y predicciones almacenadas en S3.")


## 17. Visualizar desempeño del mejor modelo

Se compara la temperatura real con la temperatura predicha.

La línea diagonal representa el escenario ideal:

```text
predicción = valor real
```

Mientras más cerca estén los puntos de esa línea, mejor será el comportamiento del modelo en este conjunto de prueba.


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    y_test,
    test_predictions,
    alpha=0.6
)

plt.xlabel("Temperatura real del transformador")
plt.ylabel("Temperatura predicha")
plt.title(f"Real vs. predicción — {best_model_name}")
plt.grid(alpha=0.2)
plt.show()


## 18. Verificación final de la estructura S3

Se revisan los tres bloques principales:

```text
data/
model/
output/
```

El objetivo es confirmar que el flujo completo produjo los objetos esperados.


In [ ]:
for prefix in ["data/", "model/", "output/"]:
    print(f"\n--- {prefix} ---")

    response = s3.list_objects_v2(
        Bucket=BUCKET,
        Prefix=prefix
    )

    for item in response.get("Contents", []):
        print(item["Key"])


# 19. Resumen ejecutivo

Este notebook implementa la primera integración real entre el desarrollo de Machine Learning y AWS.

## Flujo completado

```text
transformadores.csv
        │
        ▼
Amazon S3 / data/raw
        │
        ▼
Carga desde S3
        │
        ▼
Preparación
        │
        ├───────────────┐
        ▼               ▼
data/train         data/test
        │               │
        └───────┬───────┘
                ▼
        Entrenamiento local
                │
                ▼
        Evaluación
                │
                ▼
        Selección de modelos
                │
        ┌───────┴────────┐
        ▼                ▼
     model/           output/
    modelos          métricas
                    predicciones
```

## Lo que se evidencia

- Acceso a S3 desde SageMaker Studio.
- Uso del IAM Role sin credenciales escritas en código.
- Carga del dataset desde el entorno local hacia S3.
- Lectura posterior del dataset directamente desde S3.
- Organización de información en `raw`, `train` y `test`.
- Entrenamiento y comparación de varios algoritmos.
- Selección basada en métricas.
- Serialización de modelos.
- Almacenamiento de modelos en `model/`.
- Almacenamiento de métricas y predicciones en `output/`.

## Qué todavía NO hace este notebook

El entrenamiento se ejecuta dentro de JupyterLab.

Todavía no se utilizan:

- SageMaker Training Jobs;
- Model Registry;
- endpoints de inferencia;
- inferencia Serverless.

Esto es intencional.

### Siguiente etapa

El siguiente notebook podrá tomar los datasets preparados en S3 y ejecutar el entrenamiento mediante infraestructura administrada:

```text
S3 / data/train
S3 / data/test
        ↓
SageMaker Training Job
        ↓
Entrenamiento administrado
        ↓
Model Artifact
        ↓
S3 / model
```

Así el laboratorio avanza progresivamente:

**Machine Learning local → S3 → SageMaker Training Jobs → despliegue → inferencia.**
